12/09/2026
First version

In [1]:
from io import TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.tree import DecisionTreeClassifier


In [2]:
PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")


def partidas_en_stream(ruta: Path):
    """Genera partidas PGN una a una desde un archivo .pgn.zst."""
    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida


def partida_en_indice(ruta: Path, indice: int):
    """Devuelve la partida con índice cero-based sin cargar todas las partidas."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    for indice_actual, partida in enumerate(partidas_en_stream(ruta)):
        if indice_actual == indice:
            return partida

    raise IndexError(f"No existe una partida con índice {indice}")


# INDICE_PARTIDA = 10
# partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
# resultado_final = partida_seleccionada.headers.get("Result", "*")
# tablero = partida_seleccionada.board()
# movimientos_por_numero = {}

# for movimiento in partida_seleccionada.mainline_moves():
#     numero_movimiento = tablero.fullmove_number
#     notacion = tablero.san(movimiento)
#     movimientos_por_numero.setdefault(numero_movimiento, []).append(notacion)
#     tablero.push(movimiento)

# ultimos_10_movimientos = [
#     f"{numero}. {' '.join(movimientos)}"
#     for numero, movimientos in list(movimientos_por_numero.items())[-10:]
# ]

# print("Índice:", INDICE_PARTIDA)
# print("Resultado final:", resultado_final)
# print("Últimos 10 movimientos:")
# print(" ".join(ultimos_10_movimientos))

In [4]:
RESULTADO_A_CLASE = {
    "1-0": 0,
    "0-1": 1,
    "1/2-1/2": 2,
}

CARACTERISTICAS = [
    "reina_blancas",
    "reina_negras",
    "blancas_enrocadas",
    "negras_enrocadas",
]


def partida_a_dataframe(partida: chess.pgn.Game, movimientos_desde_final: int = 10):
    """Devuelve una fila con el estado 10 jugadas completas antes del final."""
    if movimientos_desde_final < 0:
        raise ValueError("movimientos_desde_final debe ser mayor o igual que cero")

    resultado = partida.headers.get("Result", "*")
    if resultado not in RESULTADO_A_CLASE:
        raise ValueError(f"Resultado PGN no válido para clasificación: {resultado}")

    movimientos = list(partida.mainline_moves())
    tablero = partida.board()
    estados = []
    blancas_enrocadas = False
    negras_enrocadas = False

    estados.append(
        {
            "reina_blancas": int(bool(tablero.pieces(chess.QUEEN, chess.WHITE))),
            "reina_negras": int(bool(tablero.pieces(chess.QUEEN, chess.BLACK))),
            "blancas_enrocadas": 0,
            "negras_enrocadas": 0,
        }
    )

    for movimiento in movimientos:
        if tablero.is_castling(movimiento):
            if tablero.turn == chess.WHITE:
                blancas_enrocadas = True
            else:
                negras_enrocadas = True

        tablero.push(movimiento)
        estados.append(
            {
                "reina_blancas": int(bool(tablero.pieces(chess.QUEEN, chess.WHITE))),
                "reina_negras": int(bool(tablero.pieces(chess.QUEEN, chess.BLACK))),
                "blancas_enrocadas": int(blancas_enrocadas),
                "negras_enrocadas": int(negras_enrocadas),
            }
        )

    plies_desde_final = movimientos_desde_final * 2
    indice_estado = max(0, len(estados) - 1 - plies_desde_final)
    caracteristicas = estados[indice_estado]
    caracteristicas["y"] = RESULTADO_A_CLASE[resultado]

    return pd.DataFrame([caracteristicas], columns=CARACTERISTICAS + ["y"])


INDICE_PARTIDA = 2
partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
datos_partida = partida_a_dataframe(partida_seleccionada)
X = datos_partida[CARACTERISTICAS]
y = datos_partida["y"]

print("Índice:", INDICE_PARTIDA)
print("Resultado:", partida_seleccionada.headers["Result"])
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
datos_partida


Índice: 2
Resultado: 1-0
Forma de X: (1, 4)
Forma de y: (1,)


,reina_blancas,reina_negras,blancas_enrocadas,negras_enrocadas,y
0,1,1,0,0,0
